# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [7]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [8]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Loan Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [9]:
from langchain_community.document_loaders.csv_loader import CSVLoader
from datetime import datetime, timedelta

loader = CSVLoader(
    file_path=f"./data/complaints.csv",
    metadata_columns=[
      "Date received", 
      "Product", 
      "Sub-product", 
      "Issue", 
      "Sub-issue", 
      "Consumer complaint narrative", 
      "Company public response", 
      "Company", 
      "State", 
      "ZIP code", 
      "Tags", 
      "Consumer consent provided?", 
      "Submitted via", 
      "Date sent to company", 
      "Company response to consumer", 
      "Timely response?", 
      "Consumer disputed?", 
      "Complaint ID"
    ]
)

loan_complaint_data = loader.load()

for doc in loan_complaint_data:
    doc.page_content = doc.metadata["Consumer complaint narrative"]

Let's look at an example document to see if everything worked as expected!

In [10]:
loan_complaint_data[0].page_content

"The federal student loan COVID-19 forbearance program ended in XX/XX/XXXX. However, payments were not re-amortized on my federal student loans currently serviced by Nelnet until very recently. The new payment amount that is effective starting with the XX/XX/XXXX payment will nearly double my payment from {$180.00} per month to {$360.00} per month. I'm fortunate that my current financial position allows me to be able to handle the increased payment amount, but I am sure there are likely many borrowers who are not in the same position. The re-amortization should have occurred once the forbearance ended to reduce the impact to borrowers."

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "LoanComplaints".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [87]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    loan_complaint_data,
    embeddings,
    location=":memory:",
    collection_name="LoanComplaints"
)

## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [12]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [13]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [14]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [15]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

In [16]:
naive_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans appear to be related to mismanagement and misinformation, including errors in balances, misapplied payments, wrongful denials of payment plans, and confusion caused by multiple transfers and lack of communication from servicers. Other prevalent issues include inaccuracies in loan reporting, difficulties in applying payments correctly, complicated and untransparent handling of interest and balances, and problems with repayment options such as forbearance leading to increased debt due to accumulated interest.\n\nIn summary, the most common issue with loans seems to be **mismanagement and miscommunication by lenders or servicers, leading to errors, incorrect information, and difficulties in proper repayment and account handling**.'

In [17]:
naive_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, some complaints did not get handled in a timely manner. Specifically, the complaint from the user with Complaint ID 12709087, received on 03/28/25, was marked as "Not timely" in response to the company\'s handling. The details indicate that the consumer waited over the expected response time without receiving a resolution.'

In [18]:
naive_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

"People failed to pay back their loans primarily due to a combination of factors including mismanagement of loan information, lack of clear communication from lenders or servicers, unexpected or unauthorized loan transferments, and difficulties in understanding or navigating repayment options. Many borrowers were unaware of when their repayment was expected to resume, or were not properly notified about changes to their loan status, leading to unintentional delinquencies and negative impacts on their credit scores. Additionally, issues such as accruing interest during deferment or forbearance, limited loan repayment options, and perceived predatory practices (e.g., applying payments mainly to interest, making it hard to reduce principal) contributed to borrowers struggling to manage or clear their debt. Overall, inadequate information, poor communication, and administrative errors played significant roles in borrowers' failure to successfully repay their loans."

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [19]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(loan_complaint_data)

We'll construct the same chain - only changing the retriever.

In [20]:
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [21]:
bm25_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to dealing with lenders or servicers, such as:\n\n- Disputes over fees charged\n- Trouble with how payments are being handled (e.g., applying payments incorrectly, making repayment overly complicated)\n- Receiving bad or conflicting information about loan balances or terms\n\nThese issues reflect challenges in communication, transparency, or fairness in loan servicing. Therefore, the most common issue reported seems to be problems associated with the handling and servicing of loans by lenders or servicers.'

In [22]:
bm25_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the information provided, all the complaints mentioned indicate that the companies responded with a "Closed with explanation" and the responses were marked as "Timely response? Yes." This suggests that the complaints were handled within an appropriate timeframe. Therefore, there is no evidence within the provided data that any complaints did not get handled in a timely manner.'

In [23]:
bm25_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for several reasons, including:\n\n1. Problems with payment plans or forbearances: Some borrowers had difficulty with their payment plans or were steered into wrong types of forbearances, which led to increased loan balances and difficulties in repayment.\n2. Lack of communication or awareness: Borrowers were sometimes unaware that their loans had been transferred to different servicers, or that their autopayments had been discontinued without proper notice, resulting in missed payments.\n3. Errors and bad information from servicers: There are cases where servicers provided bad or insufficient information about the loans, failed to respond to requests for deferment or forbearance, or did not update payment statuses, causing borrowers to fall behind.\n4. Technical or administrative issues: Problems such as payments being reversed due to errors, or systems not properly processing payments, contributed to missed or late payments.\n5. Administrative n

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

#### Answer #1:

I can't think on an query related to student loans, I know nothing about that, but here is an example on my domain:

This query "CVE-2024-4577" will represent a win to BM25 because:

- That is an unique identifier used on a especific realm, essentialy an opaque token that do not carry any meaning in an embedding space. Since BM25 treats every term the same, that will give us high precision.
- Embeddings will struggle with those rare/numeric tokens, probably the model not have seen the term in training, it will probably split it into subwords diluting the importance of the term.
- Same with error codes. you often need the precise advisory for a single CVE, or a solution for a especific error code. BM25’s term‐frequency/IDF scoring ensures the document that literally mentions your query is ranked highest.


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [24]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [25]:
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
contextual_compression_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issue with loans appears to be problems related to the management and handling of student loans by lenders and servicers. This includes errors in loan balances, misapplied payments, lack of clear information about loan details such as interest and payoff amounts, improper transfer of loans without consent, and issues with communication and documentation. Specifically, challenges like accumulating interest during forbearance, confusing or inconsistent account information, and inadequate disclosure of loan terms seem to be prominent issues.'

In [27]:
contextual_compression_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, there are complaints that did not get handled in a timely manner. Specifically, one complaint about a loan account request has been open for over a year without resolution, with the customer still awaiting a response despite multiple follow-ups and nearly 18 months passing. Additionally, another complaint about payments not being applied to the account indicates ongoing issues over a period of 2-3 weeks.'

In [28]:
contextual_compression_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People often fail to pay back their loans due to lack of clear and timely communication from loan servicers, changes in ownership of the loans without proper notification, and the accumulation of unpaid interest that makes repayment difficult. Additionally, many borrowers are unaware of their repayment obligations, especially when they are not properly informed about the terms of their loans, how interest accumulates, or the availability of payment plans. As a result, some borrowers find themselves in financial hardship, with balances increasing over time despite making payments, leading to difficulties in managing and repaying their loans.'

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [29]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
)

In [30]:
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [31]:
multi_query_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

"Based on the provided data, the most common issue with loans appears to be problems related to 'Dealing with your lender or servicer,' specifically issues like trouble with how payments are being handled, incorrect information on reports, and difficulty accessing accurate account information. Many complaints highlight mismanagement such as unnotified late payments, incorrect account statuses, disputes over interest calculations, and lack of transparency. \n\nTherefore, the most common issue seems to be **problems with loan servicing, including mismanagement, misinformation, and lack of communication from lenders or servicers**."

In [32]:
multi_query_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaints data, yes, there are complaints indicating that some complaints were not handled in a timely manner. Specifically:\n\n- In one case with Complaint ID 12739706 (filed by MOHELA), the response was marked as "No" for timely response, and the complaint indicates that it has been over 1 year since the initial request with no resolution, suggesting significant delays.\n- Another complaint with Complaint ID 12973003 (also MOHELA) was marked as "Timely response? Yes," but the complaint history mentions that the issue persisted for over 2-3 weeks without resolution, and the complainant was waiting for over a year for a response.\n- A different case (Complaint ID 13062402, with EdFinancial Services) was marked as "Timely response? Yes," but the complainant still reports ongoing issues and lack of resolution after months.\n- Several other complaints note repeated follow-ups, delays in responses, or that the issue persisted for months to over a year without resolu

In [33]:
multi_query_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans primarily due to a combination of systemic issues and mismanagement by loan servicers. The key reasons include:\n\n1. Lack of clear information about repayment options and associated costs, especially regarding interest accumulation during forbearance or deferment periods.\n2. Servicer misconduct, such as misapplied payments, errors in loan balances, and wrongful reporting to credit bureaus, which can adversely affect credit scores.\n3. Forbearance steering practices that misled borrowers into long-term forbearance instead of more manageable income-driven repayment plans or rehabilitation options, leading to increased debt due to interest capitalization.\n4. Poor communication and unhelpful customer service, making it difficult for borrowers to understand their repayment obligations or to access alternative payment plans.\n5. Systemic failures in the handling and reporting of loan data, often resulting in inaccurate account statuses, overlooked el

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

#### Answer #2:

Different users express the same intent using different words, a query "loan problems" might be reformulated as "loan complaints", "loan issues", etc. This increases the chance that any document containing a variant is brought back

Users and documents often use different terms for the same concept ("car" vs. "automobile", "heart attack" vs. "myocardial infarction"). Relevant documents that overs both terms will be retreived.

## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [34]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [35]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [36]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [37]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [38]:
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [39]:
parent_document_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided context, the most common issues with loans appear to involve errors or problems related to loan servicing, such as incorrect information on credit reports, misapplied payments, wrongful denials of payment plans, discrepancies in loan balances and interest rates, and issues arising from the transfer or sale of loans. Many complaints also highlight systemic failures, lack of proper verification, and unfair or deceptive practices by loan servicers.\n\nIn summary, the most common issues are related to:\n- Misreporting or incorrect information on credit reports\n- Errors in loan balances and interest rates\n- Problems with loan servicing, including misapplied payments and wrongful denials\n- Discrepancies due to loan transfers or sales\n- Unfair or deceptive practices by lenders or servicers\n\nIf I had to identify the single most common issue from this data, it would be errors or inaccuracies in loan reporting and servicing that lead to credit report inaccuracies and

In [40]:
parent_document_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided information, several complaints indicate they were not handled in a timely manner. Specifically, the complaints with Complaint IDs 12709087 and 12935889 explicitly state that responses were not timely ("Timely response?": "No"). Both of these complaints involve issues with federal student loan servicing managed by MOHELA, and the narratives detail ongoing delays and lack of communication.\n\nTherefore, yes, some complaints did not get handled in a timely manner.'

In [41]:
parent_document_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including experiencing severe financial hardship, lack of proper information or guidance about their repayment options, and issues related to the management and transparency of their loans. Specifically, some borrowers faced difficulties because their lenders resumed payments prematurely or failed to provide adequate notification or support, and others could not find employment or had to rely on deferments and forbearances, which increased the total debt due to accrued interest. Additionally, some borrowers were misled about the value of their education or the manageability of their loans, especially when their schools faced financial instability or closed, making it difficult for them to secure employment and repay their loans.'

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [42]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [43]:
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [44]:
ensemble_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'The most common issues with loans, based on the complaints provided, include:\n\n1. Dealing with lenders or servicers, including receiving bad or incorrect information about the loan, and problems with how payments are being handled.\n2. Problems with payment plans, such as being unable to apply extra payments to principal, or being steered into unfavorable forbearance or deferment options that increase interest.\n3. Inaccurate or inconsistent information on loan balances, interest calculations, or improper reporting to credit bureaus, often leading to credit score drops.\n4. Lack of proper communication and insufficient disclosure of loan terms, payment obligations, or changes in servicing.\n5. Issues related to loan transfers or mishandling, including unauthorized transfers, improper documentation, or failures to provide legally required loan documents like Master Promissory Notes.\n6. Disputes over loan legitimacy, including fraudulent or questionable origination, mismanagement, or

In [45]:
ensemble_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Yes, according to the provided complaints, several complaints indicate that complaints were not handled in a timely manner. Specifically:\n\n- On page 4, complaint with ID 12739706 (row 811) about MOHELA\'s failure to inform about delinquent loans after 90 days was marked as "timely response?": No.\n- On page 4, complaint with ID 12823876 (row 400) regarding late payments not being correctly reported was responded to in a timely manner.\n- On page 4, complaint with ID 12935889 (row 414) about loan mismanagement and credit reporting was marked as "Timely response?": No.\n- On page 4, complaint with ID 13056764 (row 418) about late payments not being removed was responded to in a timely manner.\n- On page 4, complaint with ID 13131123 (row 509) about investigation delays was marked "Timely response?": Yes, but the complaint indicates ongoing issues with delayed responses and unresolved issues.\n- Several other complaints, such as ID 12792958 (row 288), mention delays or lack of proper r

In [46]:
ensemble_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including:\n\n- Lack of proper notification about payment due dates or account status changes, leading to unawareness of repayment obligations.\n- Mismanagement or errors by loan servicers, such as incorrect account information, failure to follow regulations, or mishandling of account transfers.\n- Financial hardships and hardships exacerbated by accumulating interest, limited income, or unemployment, making payments unaffordable.\n- Lack of support or guidance from loan servicers when requesting deferments, forbearance, or payment plan adjustments, resulting in missed payments.\n- Being misled about loan forgiveness options or the actual terms of repayment.\n- Technical issues or poor communication, such as payments being reversed or not properly applying to principal, or outdated contact information preventing borrower contact.\n- Unauthorized sharing of personal information or privacy violations that impacted credit and tru

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [47]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [48]:
semantic_documents = semantic_chunker.split_documents(loan_complaint_data[:20])

Let's create a new vector store.

In [49]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Loan_Complaint_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [50]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [51]:
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [52]:
semantic_retrieval_chain.invoke({"question" : "What is the most common issue with loans?"})["response"].content

'Based on the provided data, the most common issue with loans appears to be problems related to the handling and servicing of student loans, including:\n\n- Struggling to repay loans or problems with repayment plans.\n- Errors or disputes about loan account status, such as being in default when the borrower has not been.\n- Issues with loan reporting, such as incorrect information on credit reports or improper collection practices.\n- Difficulties in communication with servicers or lenders, including lack of transparency, long wait times, and inconsistent information.\n- Problems with loan documentation, processing, or documentation submission.\n\nWhile specific issues vary, complaints about mismanagement, misreporting, and communication failures seem to be prevalent themes.'

In [53]:
semantic_retrieval_chain.invoke({"question" : "Did any complaints not get handled in a timely manner?"})["response"].content

'Based on the provided complaint records, several complaints indicate that their issues were not handled in a timely manner. Specifically, all the complaints listed show responses from the companies that were marked as "Closed with explanation" and indicate that responses were received promptly ("Timely response?": "Yes"). However, the complaints also reveal ongoing issues, such as unaddressed errors, repeated violations, and lack of follow-up, implying that initial handling or resolution was not sufficient or satisfactory.\n\nFor example:\n- The complaint about Nelnet (ID: 13331376) details multiple attempts to address misconduct and errors, with the company response marked as "Closed with explanation," despite the consumer reporting ongoing issues and misconduct.\n- Multiple other complaints highlight that while responses may have been timely, the issues were not effectively resolved, resulting in continued disputes, errors, or lack of proper correction.\n\nIn summary, while response

In [54]:
semantic_retrieval_chain.invoke({"question" : "Why did people fail to pay back their loans?"})["response"].content

'People failed to pay back their loans for various reasons, including difficulties with loan administration and communication issues, disputes over the legitimacy of the debt, improper reporting, and data breaches. Some specific causes mentioned are:\n\n- Lack of clear or accurate information from lenders or servicers about loan status or repayment terms.\n- Stalling or delays by loan servicers (e.g., EdFinancial) that discourage borrowers from continuing efforts to resolve issues.\n- Disputes over the legitimacy of the debt, such as loans being reported as defaulted or in bad standing without proper verification.\n- Errors in payment processing or technical issues, leading to missed or uncredited payments.\n- Unauthorized access or breaches of personal data related to the loans, which can complicate repayment or lead to legal and privacy concerns.\n\nIn summary, failures in communication, administrative errors, disputes over debt legitimacy, and data security breaches are contributing

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?


#### Answer #3:

Highly repetitive sentences would have very similar embeddings, that will make it difficult for the chunker to indentify good breaking points.

This would help adjust the algorithm:

```python
   def adaptive_semantic_chunker(documents, embeddings):
       # calculate the avg sentence lenght and the repetition score
       avg_sentence_length = calculate_avg_sentence_length(documents)
       repetition_score = calculate_repetition_score(documents)
       
       if repetition_score > 0.8:  # only the most dissimilar sentences will create breakepoints
           threshold = 98  
       elif avg_sentence_length < 50:  # here the effect would be a combination of short sentences into meaningful chunks
           threshold = 90 
       else:
           threshold = 85 # defaults to 85, this is a balances approach
       
       return SemanticChunker(
           embeddings,
           breakpoint_threshold_type="percentile",
           percentile_threshold=threshold
       )
```

# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

## Activiy #1
**could not make it, so please just grade my answers above**

In [61]:
# we need this

from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
import pandas as pd

In [65]:
# load the official pdfs to create a ground truth knowledge graph

path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
official_docs = loader.load()

In [68]:
# load the complaints data

df = pd.read_csv("data/complaints.csv")
queries = df["Consumer complaint narrative"].dropna().tolist()

In [79]:
# setup RAGAS
from ragas.testset.graph import KnowledgeGraph
from ragas.testset.graph import Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms

llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o", temperature=0))
emb = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
generator = TestsetGenerator(llm=llm, embedding_model=emb)


In [80]:
# knowledge graph

kg = KnowledgeGraph()

for doc in official_docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )

default_transforms = default_transforms(documents=official_docs[:20], llm=llm, embedding_model=emb)
apply_transforms(kg, default_transforms)

Applying HeadlinesExtractor:   0%|          | 0/17 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/30 [00:00<?, ?it/s]

Property 'summary' already exists in node '981523'. Skipping!
Property 'summary' already exists in node 'd5a397'. Skipping!
Property 'summary' already exists in node '5c9198'. Skipping!
Property 'summary' already exists in node '6a9f43'. Skipping!
Property 'summary' already exists in node '2dbf1b'. Skipping!
Property 'summary' already exists in node 'cdc58f'. Skipping!
Property 'summary' already exists in node '825d44'. Skipping!
Property 'summary' already exists in node '833a92'. Skipping!
Property 'summary' already exists in node '441282'. Skipping!
Property 'summary' already exists in node '2171cd'. Skipping!
Property 'summary' already exists in node 'f96a28'. Skipping!
Property 'summary' already exists in node 'c33d4d'. Skipping!
Property 'summary' already exists in node '80a471'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/42 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '2171cd'. Skipping!
Property 'summary_embedding' already exists in node 'cdc58f'. Skipping!
Property 'summary_embedding' already exists in node '2dbf1b'. Skipping!
Property 'summary_embedding' already exists in node 'c33d4d'. Skipping!
Property 'summary_embedding' already exists in node '981523'. Skipping!
Property 'summary_embedding' already exists in node '5c9198'. Skipping!
Property 'summary_embedding' already exists in node '6a9f43'. Skipping!
Property 'summary_embedding' already exists in node 'd5a397'. Skipping!
Property 'summary_embedding' already exists in node '833a92'. Skipping!
Property 'summary_embedding' already exists in node '825d44'. Skipping!
Property 'summary_embedding' already exists in node 'f96a28'. Skipping!
Property 'summary_embedding' already exists in node '80a471'. Skipping!
Property 'summary_embedding' already exists in node '441282'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

In [81]:
# save the thing

kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")

In [ ]:
# create a silver dataset
# this is supposed to be a gold dataset after manual review
# but I don't know nothing about student loans to make a judgement call

from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer


generator = TestsetGenerator(llm=llm, embedding_model=emb, knowledge_graph=loan_data_kg)

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=llm), 0.25),
]

silver = generator.generate_with_langchain_docs(official_docs[:50], testset_size=10)


Applying HeadlinesExtractor:   0%|          | 0/40 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/50 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/66 [00:00<?, ?it/s]

Property 'summary' already exists in node '3dfd74'. Skipping!
Property 'summary' already exists in node 'f23835'. Skipping!
Property 'summary' already exists in node '567f0e'. Skipping!
Property 'summary' already exists in node '83d5ff'. Skipping!
Property 'summary' already exists in node '0df5db'. Skipping!
Property 'summary' already exists in node 'b767d8'. Skipping!
Property 'summary' already exists in node '3bf97a'. Skipping!
Property 'summary' already exists in node 'e4318b'. Skipping!
Property 'summary' already exists in node '34a6dd'. Skipping!
Property 'summary' already exists in node '8118c4'. Skipping!
Property 'summary' already exists in node 'ad4922'. Skipping!
Property 'summary' already exists in node '9a78a0'. Skipping!
Property 'summary' already exists in node '9f531f'. Skipping!
Property 'summary' already exists in node '2e5eb5'. Skipping!
Property 'summary' already exists in node '80e351'. Skipping!
Property 'summary' already exists in node 'a7ff71'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/31 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/116 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'b85ee6'. Skipping!
Property 'summary_embedding' already exists in node '83d5ff'. Skipping!
Property 'summary_embedding' already exists in node 'aa088e'. Skipping!
Property 'summary_embedding' already exists in node '3dfd74'. Skipping!
Property 'summary_embedding' already exists in node 'b767d8'. Skipping!
Property 'summary_embedding' already exists in node '0df5db'. Skipping!
Property 'summary_embedding' already exists in node 'e4318b'. Skipping!
Property 'summary_embedding' already exists in node '34a6dd'. Skipping!
Property 'summary_embedding' already exists in node '78014a'. Skipping!
Property 'summary_embedding' already exists in node '567f0e'. Skipping!
Property 'summary_embedding' already exists in node 'c378b9'. Skipping!
Property 'summary_embedding' already exists in node '9f531f'. Skipping!
Property 'summary_embedding' already exists in node '8118c4'. Skipping!
Property 'summary_embedding' already exists in node '894629'. Sk

In [86]:
df_silver = silver.to_pandas()
df_silver.to_csv("silver_testset.csv", index=False)
df_silver

,user_input,reference_contexts,reference,synthesizer_name
0,How does the Department handle requests for ac...,"[Chapter 1 Academic Years, Academic Calendars,...",Schools that provide 2- or 4-year associate or...,single_hop_specifc_query_synthesizer
1,What are the key differences between standard ...,"[non-term (includes clock-hour calendars), or ...","In a standard term academic calendar, all clas...",single_hop_specifc_query_synthesizer
2,What are the exceptions to the normal loan per...,[Inclusion of Clinical Work in a Standard Term...,The exceptions to the normal loan period and d...,single_hop_specifc_query_synthesizer
3,Is the Federal Work-Study (FWS) Program subjec...,[Non-Term Characteristics A program that measu...,The payment period is applicable to all Title ...,single_hop_specifc_query_synthesizer
4,Wht is the purpse of Appendix A in the context...,[both the credit or clock hours and the weeks ...,Appendix A illustrates the principles describe...,single_hop_specifc_query_synthesizer
5,How does the inclusion of clinical work in a s...,[<1-hop>\n\nInclusion of Clinical Work in a St...,The inclusion of clinical work in a standard t...,multi_hop_abstract_query_synthesizer
6,How clinical work can be included in a standar...,[<1-hop>\n\nInclusion of Clinical Work in a St...,Clinical work can be included in a standard te...,multi_hop_abstract_query_synthesizer
7,What are the academic year requirements for st...,"[<1-hop>\n\nChapter 1 Academic Years, Academic...","For standard term programs, such as those usin...",multi_hop_abstract_query_synthesizer
8,"How does Volume 8, Chapter 3 provide guidance ...",[<1-hop>\n\nDisbursement Timing in Subscriptio...,"Volume 8, Chapter 3 provides detailed guidance...",multi_hop_specific_query_synthesizer
9,How do the principles outlined in Appendix A a...,[<1-hop>\n\nboth the credit or clock hours and...,The principles outlined in Appendix A affect t...,multi_hop_specific_query_synthesizer


In [90]:
# naive retrieval
# setting up the vector store

from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    official_docs,
    embeddings,
    location=":memory:",
    collection_name="OfficialDocs"
)

In [92]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})

In [91]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

In [93]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

In [95]:
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)